# AI-Powered Resume Screening Assistant

## 1.  Install Dependencies

In [1]:
#mount Google Drive if the project is stored there.
from google.colab import drive
drive.mount('/content/drive')


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b2-hyv4e68gh14s?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_log

In [2]:
!pip install kagglehub pandas sentence-transformers qdrant-client
!pip install faiss-gpu-cu12 sentence-transformers transformers accelerate \ gradio bitsandbytes einops --quiet
import os
import re
import gc
import json
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict, Tuple, Optional

import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
import gradio as gr

warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00:00:0100:01


## 2. Dowload dataset

### Resume dataset

In [13]:
# ── Load Resume Dataset ──────────────────────────────────────────────────────────────
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'resume-dataset' dataset.
Path to dataset files: /kaggle/input/resume-dataset


In [24]:
from pathlib import Path
import pandas as pd

data_dir = Path("/kaggle/input/resume-dataset")

csv_files = list(data_dir.rglob("*.csv"))
print(csv_files)

df = pd.read_csv(csv_files[0])
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
df.head(3)

print("Resume categories:")
print(df['Category'].value_counts().to_string())

[PosixPath('/kaggle/input/resume-dataset/Resume/Resume.csv')]
Shape   : (2484, 4)
Columns : ['ID', 'Resume_str', 'Resume_html', 'Category']
Resume categories:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22


### Job description dataset

In [ ]:
# ── Load Job  Dataset ──────────────────────────────────────────────────────────────
import kagglehub

# Download latest version
path = kagglehub.dataset_download("batuhanmutlu/job-skill-set")

print("Path to dataset files:", path)

100%|██████████| 1.50M/1.50M [00:00<00:00, 90.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/batuhanmutlu/job-skill-set/versions/2


In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("/root/.cache/kagglehub/datasets/batuhanmutlu/job-skill-set/versions/2")

csv_files = list(data_dir.rglob("*.csv"))
print(csv_files)

df = pd.read_csv(csv_files[0])
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
df.head(3)

print("Resume categories:")
print(df['category'].value_counts().to_string())

[PosixPath('/root/.cache/kagglehub/datasets/batuhanmutlu/job-skill-set/versions/2/all_job_post.csv')]
Shape   : (1167, 5)
Columns : ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set']
Resume categories:
category
INFORMATION-TECHNOLOGY    240
BUSINESS-DEVELOPMENT      239
FINANCE                   236
SALES                     232
HR                        220


## 3. Preprocessing

Result:
Returns a cleaned text string containing only letters, digits, spaces, and basic punctuation (periods, commas, semicolons, colons, hyphens, slashes, parentheses). All URLs, email addresses, phone numbers, special characters (e.g., !@#$%^&*+=<>[]{}) are removed, and extra whitespace (including newlines) is collapsed into single spaces.

### resume dataset

In [25]:
def clean_resume(text: str) -> str:
    """Clean raw resume text."""
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    # Remove phone numbers
    text = re.sub(r'[\+\(]?[1-9][0-9 .\-\(\)]{8,}[0-9]', ' ', text)
    # Remove special characters but keep letters, digits, spaces, basic punctuation
    text = re.sub(r'[^\w\s\.,;:\-\/\(\)]', ' ', text)
    # Collapse multiple spaces / newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Use whichever column holds the full resume text
TEXT_COL = 'Resume_str' if 'Resume_str' in df.columns else 'Resume'

df = df[['Category', TEXT_COL]].copy()
df.columns = ['category', 'raw_text']
df['text'] = df['raw_text'].apply(clean_resume)

# Drop raw_text column
df = df.drop(columns=["raw_text"])

# Drop resumes that are too short after cleaning
df = df[df['text'].str.len() >= 200].reset_index(drop=True)

print(f"Total resumes after cleaning : {len(df)}")
print(f"Avg text length              : {df['text'].str.len().mean():.0f} chars")
print(f"Categories                   : {df['category'].nunique()}")
df.head(3)

Total resumes after cleaning : 2483
Avg text length              : 5903 chars
Categories                   : 24


,category,text
0,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINI...
1,HR,"HR SPECIALIST, US HR OPERATIONS Summary Versat..."
2,HR,HR DIRECTOR Summary Over 20 years experience i...


Split dataset

In [26]:
from sklearn.model_selection import train_test_split

# Check category counts first
category_counts = df["category"].value_counts()
rare_categories = category_counts[category_counts < 7]

if len(rare_categories) > 0:
    print("Warning: Some categories have fewer than 7 samples.")
    print("These categories may not appear in all train/validation/test splits:")
    print(rare_categories.to_string())

# 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["category"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["category"]
)

# Check category coverage
all_categories = set(df["category"].unique())

for name, split_df in {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}.items():
    split_categories = set(split_df["category"].unique())
    missing = sorted(all_categories - split_categories)

    print(f"\n{name.upper()} size: {len(split_df)}")
    print(f"Categories: {len(split_categories)} / {len(all_categories)}")

    if missing:
        print("Missing categories:", missing)
    else:
        print("All categories are included.")


TRAIN size: 1738
Categories: 24 / 24
All categories are included.

VALIDATION size: 372
Categories: 24 / 24
All categories are included.

TEST size: 373
Categories: 24 / 24
All categories are included.


In [27]:
print(f"Total resumes after cleaning : {len(train_df)}")
print(f"Avg text length              : {train_df['text'].str.len().mean():.0f} chars")
print(f"Categories                   : {train_df['category'].nunique()}")
print(train_df['category'].value_counts().to_string())
train_df[['category', 'text']].head(3)

Total resumes after cleaning : 1738
Avg text length              : 5915 chars
Categories                   : 24
category
INFORMATION-TECHNOLOGY    84
CHEF                      83
ENGINEERING               83
ACCOUNTANT                83
ADVOCATE                  83
BUSINESS-DEVELOPMENT      83
FINANCE                   83
FITNESS                   82
AVIATION                  82
SALES                     81
CONSULTANT                81
HEALTHCARE                80
BANKING                   80
PUBLIC-RELATIONS          78
CONSTRUCTION              78
HR                        77
DESIGNER                  75
ARTS                      72
TEACHER                   71
APPAREL                   68
DIGITAL-MEDIA             67
AGRICULTURE               44
AUTOMOBILE                25
BPO                       15


,category,text
1012,SALES,SALES Summary Focused and dedicated insurance ...
1425,CHEF,"Summary Sous Chef, Lead Cook and Supervisor wi..."
2018,CONSTRUCTION,LICENSE CONTRACTOR Summary Detail-oriented spe...


In [18]:
print(f"Total resumes after cleaning : {len(val_df)}")
print(f"Avg text length              : {val_df['text'].str.len().mean():.0f} chars")
print(f"Categories                   : {val_df['category'].nunique()}")
print(val_df['category'].value_counts().to_string())
val_df[['category', 'text']].head(3)

Total resumes after cleaning : 372
Avg text length              : 5958 chars
Categories                   : 24
category
BUSINESS-DEVELOPMENT      18
INFORMATION-TECHNOLOGY    18
FITNESS                   17
BANKING                   17
SALES                     17
AVIATION                  17
CHEF                      17
HR                        17
ENGINEERING               17
FINANCE                   17
ACCOUNTANT                17
CONSULTANT                17
PUBLIC-RELATIONS          17
HEALTHCARE                17
CONSTRUCTION              17
ADVOCATE                  17
ARTS                      16
TEACHER                   16
DESIGNER                  16
APPAREL                   15
DIGITAL-MEDIA             15
AGRICULTURE               10
AUTOMOBILE                 6
BPO                        4


,category,text
2196,BANKING,PROGRAM ASSISTANT Professional Summary Program...
1608,APPAREL,FREELANCE PRODUCTION MANAGER - MEN S WOMEN S W...
60,HR,HR COORDINATOR Summary To obtain a Human Resou...


In [28]:
print(f"Total resumes after cleaning : {len(test_df)}")
print(f"Avg text length              : {test_df['text'].str.len().mean():.0f} chars")
print(f"Categories                   : {test_df['category'].nunique()}")
print(test_df['category'].value_counts().to_string())
test_df.head(3)

Total resumes after cleaning : 373
Avg text length              : 5789 chars
Categories                   : 24
category
HEALTHCARE                18
AVIATION                  18
BUSINESS-DEVELOPMENT      18
ACCOUNTANT                18
ENGINEERING               18
FINANCE                   18
SALES                     18
INFORMATION-TECHNOLOGY    18
ADVOCATE                  18
BANKING                   18
CHEF                      18
FITNESS                   18
CONSULTANT                17
CONSTRUCTION              17
HR                        16
PUBLIC-RELATIONS          16
DESIGNER                  16
ARTS                      15
TEACHER                   15
APPAREL                   14
DIGITAL-MEDIA             14
AGRICULTURE                9
AUTOMOBILE                 5
BPO                        3


,category,text
1674,APPAREL,CFO ASSISTANT/EXECUTIVE ADMINISTRATOR/HR MANAG...
783,HEALTHCARE,OWNER Summary Results-oriented individual with...
1928,CONSTRUCTION,CONCRETE CONSTRUCTION Summary A highly experie...


Export to excel file

In [30]:
train_df.to_excel("train_resumes.xlsx", index=False)
val_df.to_excel("validation_resumes.xlsx", index=False)
test_df.to_excel("test_resumes.xlsx", index=False)

print("Excel files exported successfully.")

Excel files exported successfully.


### job description dataset

In [ ]:
import re
import html
import unicodedata
import pandas as pd


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text or "")


def strip_html(text: str) -> str:
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    return text


def clean_job_description(text: str, lowercase: bool = False) -> str:
    """
    Clean raw job description text while preserving important technical tokens.

    This function keeps useful symbols for skills such as:
    C++, C#, .NET, Node.js, React.js, SQL, HTML/CSS.
    """
    if not isinstance(text, str):
        return ""

    text = normalize_unicode(strip_html(text))

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove phone numbers
    text = re.sub(r"[\+\(]?[1-9][0-9 .\-\(\)]{8,}[0-9]", " ", text)

    # Replace null characters
    text = text.replace("\x00", " ")

    # Keep letters, digits, whitespace, and symbols useful for job skills
    text = re.sub(r"[^\w\s.,;:()&/#%+\-@.]", " ", text)

    # Normalize repeated separators
    text = re.sub(r"[-=_]{4,}", " ", text)

    # Fix spacing around punctuation
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"([,.;:])(?=\S)", r"\1 ", text)

    # Collapse spaces and newlines
    text = re.sub(r"\s+", " ", text).strip()

    if lowercase:
        text = text.lower()

    return text


required_cols = ["job_id", "category", "job_title", "job_description", "job_skill_set"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}. Available columns: {df.columns.tolist()}")

df = df[required_cols].copy()

df["job_title"] = df["job_title"].apply(clean_job_description)
df["job_description"] = df["job_description"].apply(lambda x: clean_job_description(x, lowercase=True))

df["job_skill_set"] = df["job_skill_set"].apply(clean_job_description)

df.head(3)

df.to_csv("job_description.csv", index=False, encoding="utf-8")

In [ ]:
df.head(3)

,job_id,category,job_title,job_description,job_skill_set
0,3902668440,HR,Sr Human Resource Generalist,summary the sr. hr generalist provides hr expe...,"employee relations, talent acquisition, perfor..."
1,3905823748,HR,Human Resources Manager,be part of a stellar team at ysb as the manage...,"Talent Acquisition, Employee Performance Manag..."
2,3905854799,HR,Director of Human Resources,our client is a thriving organization offering...,"Human Resources Management, Recruitment, Emplo..."


##4. Chunking

In [ ]:
import nltk
from typing import List
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

CHUNK_SIZE = 400   # approximate chars or ~100-150 tokens
CHUNK_OVERLAP = 2  # number of sentences overlap


def chunk_text_semantic(text: str,
                        chunk_size: int = CHUNK_SIZE,
                        overlap: int = CHUNK_OVERLAP) -> List[str]:
    """
    Semantic RAG chunking:
    - sentence-based splitting
    - packs sentences into coherent chunks
    - overlaps by full sentences (not characters)
    """

    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []
    current_length = 0

    i = 0
    while i < len(sentences):

        sentence = sentences[i]
        sentence_len = len(sentence)

        # if adding sentence exceeds chunk size → close chunk
        if current_length + sentence_len > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))

            # overlap logic (keep last N sentences)
            current_chunk = current_chunk[-overlap:] if overlap > 0 else []
            current_length = sum(len(s) for s in current_chunk)

        current_chunk.append(sentence)
        current_length += sentence_len
        i += 1

    # add last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    # filter small chunks
    return [c.strip() for c in chunks if len(c) > 50]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### resume dataset chunk

In [ ]:
if "chunks_df" not in globals():
    chunks_df = pd.read_csv("resume_chunks.csv") if os.path.exists("resume_chunks.csv") else None

records = []

for _, row in df.iterrows():
    chunks = chunk_text_semantic(row["text"])
    for i, chunk in enumerate(chunks):
        records.append({
            "chunk_id": f"{row.name}_{i}",
            "resume_id": row.name,
            "category": row.get("category", "Unknown"),
            "chunk_index": i,
            "text": chunk
        })

chunks_df = pd.DataFrame(records)

print("Total chunks:", len(chunks_df))
chunks_df.to_csv("resume_chunks.csv", index=False)

Total chunks: 67099


### job description dataset chunk

In [ ]:
import os
import pandas as pd
if "chunks_df" not in globals():
    chunks_df = pd.read_csv("job_description.csv") if os.path.exists("job_description.csv") else None

records = []

for _, row in df.iterrows():
    chunks = chunk_text_semantic(row["job_description"])
    for i, chunk in enumerate(chunks):
        records.append({
            "chunk_id": f"{row["job_id"]}_{i}",
            "chunk_index": i,
            "category": row.get("category", "Unknown"),
            "job_id": row["job_id"],
            "job_title": row["job_title"],
            "job_skill_set": row["job_skill_set"],
            "job_description": chunk
        })

chunks_df = pd.DataFrame(records)

print("Total chunks:", len(chunks_df))
chunks_df.to_csv("job_description_chunks.csv", index=False)

Total chunks: 21734


## Chunking for job

##5. Embedding

In [ ]:
# EMBEDDING_MODEL = 'multi-qa-MiniLM-L6-cos-v1'

# embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
# print(f"Embedding model : {EMBEDDING_MODEL}")
# print(f"Embedding dim   : {embedder.get_sentence_embedding_dimension()}")

In [ ]:
from sentence_transformers import SentenceTransformer

texts = chunks_df["text"].tolist()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

KeyError: 'text'

In [ ]:
import pickle

data_to_save = {
    "texts": texts,
    "categories": chunks_df["category"].tolist(),
    "resume_ids": chunks_df["resume_id"].tolist(),
    "chunk_ids": chunks_df["chunk_id"].tolist(),
    "chunk_indexes": chunks_df["chunk_index"].tolist(),
    "embeddings": embeddings
}

with open("resume_chunk_embeddings.pkl", "wb") as f:
    pickle.dump(data_to_save, f)

print("Saved resume_chunk_embeddings.pkl")

NameError: name 'texts' is not defined

In [ ]:
from google.colab import files
files.download("resume_chunk_embeddings.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("resume_chunks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Job description dataset

In [ ]:
from sentence_transformers import SentenceTransformer

texts = chunks_df["job_description"].tolist()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/680 [00:00<?, ?it/s]

(21734, 384)


In [ ]:
import pickle

data_to_save = {
    "texts": texts,
    "job_id": chunks_df["job_id"].tolist(),
    "chunk_id": chunks_df["chunk_id"].tolist(),
    "chunk_index": chunks_df["chunk_index"].tolist(),
    "job_title": chunks_df["job_title"].tolist(),
    "category": chunks_df["category"].tolist(),
    "required_skills": chunks_df["job_skill_set"].tolist(),
    "job_description": chunks_df["job_description"].tolist(),
    "embeddings": embeddings,
}

with open("job_description_chunk_embeddings.pkl", "wb") as f:
    pickle.dump(data_to_save, f)

print("Saved job_description_chunk_embeddings.pkl")

Saved job_description_chunk_embeddings.pkl


##5. Store in Qdrant

Running on VScode



# Evaluation Dataset

In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Directly download Kaggle dataset:
# https://www.kaggle.com/datasets/surendra365/recruitement-dataset
#
# Output format:
# candidate_name, resume_id, job_id, resume_text, job_description, lable
# ============================================================

!pip install -q kagglehub pandas openpyxl

import os
import glob
import pandas as pd
import kagglehub

# ============================================================
# 1. Download dataset directly from Kaggle
# ============================================================

dataset_path = kagglehub.dataset_download("surendra365/recruitement-dataset")

print("Dataset downloaded to:")
print(dataset_path)

print("\nFiles:")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

# ============================================================
# 2. Load CSV or Excel file automatically
# ============================================================

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
xlsx_files = glob.glob(os.path.join(dataset_path, "**", "*.xlsx"), recursive=True)

if len(csv_files) > 0:
    data_path = csv_files[0]
    df = pd.read_csv(data_path)
elif len(xlsx_files) > 0:
    data_path = xlsx_files[0]
    df = pd.read_excel(data_path)
else:
    raise FileNotFoundError("No CSV or Excel file found in the dataset.")

print("\nUsing file:")
print(data_path)

print("\nOriginal shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())

display(df.head())

# ============================================================
# 3. Helper functions to detect columns
# ============================================================

def normalize_col_name(col):
    return str(col).strip().lower().replace("_", " ")

def find_column(df, possible_names):
    normalized_columns = {
        normalize_col_name(col): col
        for col in df.columns
    }

    for name in possible_names:
        name = normalize_col_name(name)
        if name in normalized_columns:
            return normalized_columns[name]

    return None

candidate_col = find_column(df, [
    "Job Applicant Name",
    "Candidate Name",
    "Applicant Name",
    "Name"
])

resume_col = find_column(df, [
    "Resume",
    "Resume Text",
    "CV",
    "CV Text"
])

job_id_col = find_column(df, [
    "Job Roles"
])

job_role_col = find_column(df, [
    "Job Roles",
    "Job Role",
    "Role",
    "Job Title",
    "Position"
])

jd_col = find_column(df, [
    "Job Description",
    "JD",
    "Description",
    "Job_Description"
])

label_col = find_column(df, [
    "Best Match",
    "Label",
    "Lable",
    "Match",
    "Matched Score",
    "matched_score",
    "Score"
])

print("\nDetected columns:")
print("candidate_col:", candidate_col)
print("resume_col:", resume_col)
print("job_id_col:", job_id_col)
print("job_role_col:", job_role_col)
print("jd_col:", jd_col)
print("label_col:", label_col)

required_columns = {
    "candidate_name": candidate_col,
    "resume_text": resume_col,
    "job_description": jd_col,
    "lable": label_col
}

missing = [name for name, col in required_columns.items() if col is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# ============================================================
# 4. Convert to required format
# ============================================================

formatted_df = pd.DataFrame()

formatted_df["candidate_name"] = df[candidate_col].astype(str).str.strip()

# Generate resume_id if not available
resume_id_col = find_column(df, [
    "Resume ID",
    "Resume_Id",
    "resume_id",
    "CV ID",
    "Candidate ID",
    "Applicant ID"
])

if resume_id_col is not None:
    formatted_df["resume_id"] = df[resume_id_col].astype(str).str.strip()
else:
    formatted_df["resume_id"] = [
        f"RES_{i:06d}" for i in range(1, len(df) + 1)
    ]

# Use original job_id if available, otherwise generate from job role + JD
if job_id_col is not None:
    formatted_df["job_id"] = df[job_id_col].astype(str).str.strip()
else:
    if job_role_col is not None:
        job_key = (
            df[job_role_col].astype(str).str.strip()
            + " || "
            + df[jd_col].astype(str).str.strip()
        )
    else:
        job_key = df[jd_col].astype(str).str.strip()

    formatted_df["job_id"] = pd.factorize(job_key)[0] + 1
    formatted_df["job_id"] = formatted_df["job_id"].apply(
        lambda x: f"JOB_{x:05d}"
    )

formatted_df["resume_text"] = df[resume_col].astype(str).str.strip()
formatted_df["job_description"] = df[jd_col].astype(str).str.strip()

# Keep spelling as "lable" because you requested this format
formatted_df["lable"] = df[label_col]

# Remove empty rows
formatted_df = formatted_df[
    (formatted_df["resume_text"].str.len() > 0)
    & (formatted_df["job_description"].str.len() > 0)
].reset_index(drop=True)

formatted_df = formatted_df[
    [
        "candidate_name",
        "resume_id",
        "job_id",
        "resume_text",
        "job_description",
        "lable"
    ]
]

print("\nFormatted shape:", formatted_df.shape)
display(formatted_df.head())

# ============================================================
# 5. Save formatted dataset
# ============================================================

output_dir = "/content/data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "test_pairs_1.csv")

formatted_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nSaved file:")
print(output_path)

# ============================================================
# 6. Download final CSV to your computer
# ============================================================

from google.colab import files
files.download(output_path)

Using Colab cache for faster access to the 'recruitement-dataset' dataset.
Dataset downloaded to:
/kaggle/input/recruitement-dataset

Files:
/kaggle/input/recruitement-dataset/job_applicant_dataset.csv

Using file:
/kaggle/input/recruitement-dataset/job_applicant_dataset.csv

Original shape: (10000, 9)

Original columns:
['Job Applicant Name', 'Age', 'Gender', 'Race', 'Ethnicity', 'Resume', 'Job Roles', 'Job Description', 'Best Match']


,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
0,Daisuke Mori,29,Male,Mongoloid/Asian,Vietnamese,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,Taichi Shimizu,31,Male,Mongoloid/Asian,Filipino,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,Sarah Martin,46,Female,White/Caucasian,Dutch,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0
3,Keith Hughes,43,Male,Negroid/Black,Caribbean,"Proficient in Budgeting, Supply Chain Optimiza...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1
4,James Davis,49,Male,White/Caucasian,English,"Proficient in Logistics, Negotiation, Procurem...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1



Detected columns:
candidate_col: Job Applicant Name
resume_col: Resume
job_id_col: Job Roles
job_role_col: Job Roles
jd_col: Job Description
label_col: Best Match

Formatted shape: (10000, 6)


,candidate_name,resume_id,job_id,resume_text,job_description,lable
0,Daisuke Mori,RES_000001,Fitness Coach,"Proficient in Injury Prevention, Motivation, N...",A Fitness Coach is responsible for helping cli...,0
1,Taichi Shimizu,RES_000002,Physician,"Proficient in Healthcare, Pharmacology, Medica...","Diagnose and treat illnesses, prescribe medica...",0
2,Sarah Martin,RES_000003,Financial Analyst,"Proficient in Forecasting, Financial Modelling...","As a Financial Analyst, you will be responsibl...",0
3,Keith Hughes,RES_000004,Supply Chain Manager,"Proficient in Budgeting, Supply Chain Optimiza...",A Supply Chain Manager oversees the entire sup...,1
4,James Davis,RES_000005,Supply Chain Manager,"Proficient in Logistics, Negotiation, Procurem...",A Supply Chain Manager oversees the entire sup...,1



Saved file:
/content/data/processed/test_pairs_1.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#TF_IDF

In [ ]:
import re
import html
import unicodedata
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def strip_html(text: str) -> str:
    text = html.unescape(text)
    return re.sub(r"<[^>]+>", " ", text)


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text)


def clean_text(text: str, preserve_case: bool = False) -> str:
    if text is None:
        return ""

    text = normalize_unicode(strip_html(str(text)))
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    if not preserve_case:
        text = text.lower()

    return text.strip()


def clean_for_matching(text: str) -> str:
    text = clean_text(text, preserve_case=False)

    # Giữ lại chữ, số, dấu +, #, ., - vì CV có C++, C#, .NET, scikit-learn, Node.js
    text = re.sub(r"[^a-zA-Z0-9+#.\-\s]", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text)

    return text.strip()


class TfidfMatcher:
    def __init__(self, ngram_range=(1, 2), max_features=30000):
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.name = "TF-IDF + Cosine Similarity"

    def score(self, jd_text, resume_texts):
        if not resume_texts:
            return []

        corpus = [clean_for_matching(jd_text)] + [
            clean_for_matching(text) for text in resume_texts
        ]

        if not any(text.strip() for text in corpus):
            return [0.0 for _ in resume_texts]

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=self.ngram_range,
            max_features=self.max_features,
            min_df=1,
        )

        matrix = vectorizer.fit_transform(corpus)
        scores = cosine_similarity(matrix[0:1], matrix[1:]).flatten()

        return np.clip(scores, 0.0, 1.0).astype(float).tolist()

In [ ]:
jd_text = """
Machine Learning Engineer with 3 years experience. Required Python SQL scikit-learn PyTorch natural language processing LangChain REST API Docker AWS data visualization bachelor computer science AWS certification.
"""

resume_texts = [
    """
    Machine learning engineer with 4 years experience. Skills Python SQL PyTorch natural language processing LangChain Docker AWS scikit-learn data visualization. Bachelor Computer Science. AWS Certified.
    """
]

matcher = TfidfMatcher()
scores = matcher.score(jd_text, resume_texts)

for i, score in enumerate(scores, start=1):
    print(f"Resume {i}: {score:.4f} = {score * 100:.2f}%")

Resume 1: 0.6493 = 64.93%


#Word2vec

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

word2vec_path = "/content/drive/MyDrive/thesis/embeddings/GoogleNews-vectors-negative300.bin"

print(os.path.exists(word2vec_path))
print(word2vec_path)

True
/content/drive/MyDrive/thesis/enbeddings/GoogleNews-vectors-negative300.bin


In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 36.0 MB/s eta 0:00:00


In [ ]:
import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from gensim.utils import simple_preprocess

word2vec_path = "/content/drive/MyDrive/thesis/embeddings/GoogleNews-vectors-negative300.bin"

model = KeyedVectors.load_word2vec_format(
    word2vec_path,
    binary=True
)

print("Vector size:", model.vector_size)
print("Vocabulary size:", len(model.key_to_index))

print(model.similarity("king", "queen"))

Vector size: 300
Vocabulary size: 3000000
0.6510957


In [ ]:
from __future__ import annotations

import html
import re
import unicodedata
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text or "")


def strip_html(text: str) -> str:
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    return text


def clean_text(text: str, preserve_case: bool = False) -> str:
    """Clean noisy resume or JD text while preserving technical tokens."""
    text = normalize_unicode(strip_html(text))
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = re.sub(r"[-=_]{4,}", " ", text)
    text = re.sub(r"[^\w\s.,;:()&/#%+\-@]", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"([,.;:])(?=\S)", r"\1 ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s+", "\n", text)
    text = text.strip()
    return text if preserve_case else text.lower()


def clean_for_display(text: str) -> str:
    return clean_text(text, preserve_case=True)


def clean_for_matching(text: str) -> str:
    return clean_text(text, preserve_case=False)


def tokenize_words(text: str) -> list[str]:
    text = clean_for_matching(text)
    return re.findall(r"[a-zA-Z0-9+#.]+", text)




def tokenize_like_vscode(text):
    # clean_for_matching
    cleaned_text = clean_for_matching(str(text))
    # dùng cùng regex
    tokens = re.findall(r"[a-zA-Z][a-zA-Z0-9+#.\-]*", cleaned_text)
    return [t for t in tokens if len(t.strip()) >= 2]

def avg_feature_vector(sentence, model, num_features=300):
    words = tokenize_like_vscode(str(sentence))
    feature_vec = np.zeros((num_features,), dtype="float32")
    n_words = 0

    for word in words:
        # Thử các dạng casing như VSCode
        for candidate in [word, word.lower(), word.title(), word.upper()]:
            if candidate in model.key_to_index:
                n_words += 1
                feature_vec += model[candidate]
                break  # chỉ lấy 1 lần

    if n_words > 0:
        feature_vec = feature_vec / n_words
    return feature_vec


In [ ]:
jd_text = "A Fitness Coach is responsible for helping clients achieve their fitness goals by designing and leading group or individual fitness programs. You will provide instruction on exercises, proper form, and injury prevention techniques, encouraging clients to push their limits while maintaining a focus on their well-being. The role requires a passion for health and fitness, a strong understanding of exercise physiology, and the ability to motivate and inspire others. You will also monitor clients’ progress and make adjustments to their fitness plans as needed to ensure continuous improvement."
resume_text = "Proficient in Injury Prevention, Motivation, Nutrition, Health Coaching, Strength Training, with mid-level experience in the field. Holds a Bachelors degree. Holds certifications such as Certified Personal Trainer (CPT) by NASM. Skilled in delivering results and adapting to dynamic environments."

jd_vec = avg_feature_vector(jd_text, model)
resume_vec = avg_feature_vector(resume_text, model)

score = cosine_similarity(
    jd_vec.reshape(1, -1),
    resume_vec.reshape(1, -1)
)[0][0]

score = float(np.clip(score, 0.0, 1.0))

print("Matching score:", score)
print("JD length:", len(jd_text))
print("Resume length:", len(resume_text))
print("JD preview:", repr(jd_text[:300]))
print("Resume preview:", repr(resume_text[:300]))
print("Vector size:", model.vector_size)
print("Vocabulary size:", len(model.key_to_index))
print("king queen:", model.similarity("king", "queen"))
print("fitness first 5:", model["fitness"][:5] if "fitness" in model else "missing")
print("Fitness first 5:", model["Fitness"][:5] if "Fitness" in model else "missing")

Matching score: 0.8109837770462036
JD length: 594
Resume length: 296
JD preview: 'A Fitness Coach is responsible for helping clients achieve their fitness goals by designing and leading group or individual fitness programs. You will provide instruction on exercises, proper form, and injury prevention techniques, encouraging clients to push their limits while maintaining a focus o'
Resume preview: 'Proficient in Injury Prevention, Motivation, Nutrition, Health Coaching, Strength Training, with mid-level experience in the field. Holds a Bachelors degree. Holds certifications such as Certified Personal Trainer (CPT) by NASM. Skilled in delivering results and adapting to dynamic environments.'
Vector size: 300
Vocabulary size: 3000000
king queen: 0.6510957
fitness first 5: [0.00692749 0.18652344 0.16894531 0.02185059 0.29101562]
Fitness first 5: [-0.0100708  -0.0559082   0.40039062 -0.06542969  0.23730469]


#Glove

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any

import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

from config.settings import DEFAULT_EMBEDDING_MODELS, STATIC_EMBEDDING_PATHS
from src.preprocessing.text_cleaner import clean_for_matching


PROJECT_ROOT = Path(__file__).resolve().parents[2]

DEFAULT_GLOVE_PATH = (
    PROJECT_ROOT / "data" / "embeddings" / "glove.6B.100d.txt"
)


@lru_cache(maxsize=1)
def load_local_glove_vectors(vector_path: str | None = None) -> Any:
    """
    Load GloVe vectors from a local .txt file.

    Expected GloVe format:
        word value1 value2 value3 ...

    Example:
        python 0.123 -0.245 0.019 ...

    Note:
    - GloVe .txt files usually do not have a word2vec-style header.
    - Therefore, we use no_header=True.
    """
    path = Path(vector_path).expanduser() if vector_path else DEFAULT_GLOVE_PATH

    if not path.is_absolute():
        path = PROJECT_ROOT / path

    if not path.exists():
        raise FileNotFoundError(f"GloVe file not found: {path}")

    print(f"Loading GloVe vectors from: {path}")

    return KeyedVectors.load_word2vec_format(
        str(path),
        binary=False,
        no_header=True,
    )


@dataclass
class GloveMatcher:
    """
    GloVe baseline matcher for resume-job description matching.

    Matching direction:
    - resume_text: one CV/resume
    - job_texts: multiple job descriptions

    Algorithm:
    1. Clean resume and job description text using clean_for_matching().
    2. Tokenize text using regex.
    3. Convert each valid token into a pretrained GloVe vector.
    4. Average all valid token vectors into one document vector.
    5. Compute cosine similarity between resume vector and each job vector.

    Output:
    - A score in the range [0, 1].
    - Higher score means the job description is more relevant to the CV.
    """

    model_id: str = DEFAULT_EMBEDDING_MODELS.get(
        "glove",
        "glove.6B.100d",
    )

    name: str = "GloVe Average Embeddings"

    vector_path: str | None = STATIC_EMBEDDING_PATHS.get("glove")

    min_token_length: int = 2

    def score(self, resume_text: str, job_texts: list[str]) -> list[float]:
        """
        Compute matching scores between one resume and many job descriptions.

        Args:
            resume_text: One CV/resume text.
            job_texts: List of job description texts.

        Returns:
            List of similarity scores. Each score is in [0, 1].
            The order of scores follows the order of job_texts.
        """
        if not job_texts:
            return []

        vectors = load_local_glove_vectors(self.vector_path)

        resume_vector = self._document_vector(resume_text, vectors)

        if resume_vector is None:
            return [0.0 for _ in job_texts]

        scores: list[float] = []

        for job_text in job_texts:
            job_vector = self._document_vector(job_text, vectors)

            if job_vector is None:
                scores.append(0.0)
                continue

            raw_score = cosine_similarity(
                resume_vector.reshape(1, -1),
                job_vector.reshape(1, -1),
            )[0][0]

            score = float(np.clip(raw_score, 0.0, 1.0))
            scores.append(score)

        return scores

    def _document_vector(self, text: str, vectors: Any) -> np.ndarray | None:
        """
        Convert a document into a single vector by mean-pooling GloVe token vectors.

        If no valid token vectors are found, return None.
        """
        tokens = self._tokenize(text)

        token_vectors: list[np.ndarray] = []

        for token in tokens:
            vector = self._lookup_vector(token, vectors)

            if vector is None:
                continue

            token_vectors.append(vector)

        if not token_vectors:
            return None

        document_vector = np.mean(
            np.vstack(token_vectors),
            axis=0,
        ).astype(np.float32)

        if np.linalg.norm(document_vector) == 0:
            return None

        return document_vector

    def _tokenize(self, text: str) -> list[str]:
        """
        Clean text and extract candidate tokens likely to exist in the GloVe vocabulary.

        This uses the same preprocessing style as Word2VecMatcher:
        - clean_for_matching()
        - regex-based token extraction
        - minimum token length filtering
        """
        cleaned_text = clean_for_matching(str(text))

        tokens = re.findall(
            r"[a-zA-Z][a-zA-Z0-9+#.\-]*",
            cleaned_text,
        )

        return [
            token
            for token in tokens
            if len(token.strip()) >= self.min_token_length
        ]

    def _lookup_vector(self, token: str, vectors: Any) -> np.ndarray | None:
        """
        Look up a token vector using multiple candidate forms.

        GloVe vocabulary is usually lowercase, but this method keeps the same
        robust lookup style as Word2VecMatcher.
        """
        for candidate in self._token_candidates(token):
            if candidate in vectors:
                return np.asarray(vectors[candidate], dtype=np.float32)

        return None

    @staticmethod
    def _token_candidates(token: str) -> list[str]:
        """
        Generate possible token forms for pretrained vector lookup.
        """
        token = str(token).strip()

        candidates = [
            token,
            token.lower(),
            token.title(),
            token.upper(),
        ]

        normalized_variants = {
            "c++": "cpp",
            "c#": "csharp",
            ".net": "dotnet",
            "node.js": "nodejs",
            "react.js": "reactjs",
            "vue.js": "vuejs",
            "next.js": "nextjs",
            "javascript": "javascript",
            "js": "javascript",
            "typescript": "typescript",
            "ts": "typescript",
        }

        lower = token.lower()

        if lower in normalized_variants:
            candidates.append(normalized_variants[lower])

        unique_candidates: list[str] = []

        for candidate in candidates:
            if candidate and candidate not in unique_candidates:
                unique_candidates.append(candidate)

        return unique_candidates

    @staticmethod
    def _cosine_similarity_01(a: np.ndarray, b: np.ndarray) -> float:
        """
        Compute cosine similarity and clip it to [0, 1].

        Do not use (cosine + 1) / 2 because it artificially inflates weak matches.
        """
        denominator = np.linalg.norm(a) * np.linalg.norm(b)

        if denominator == 0:
            return 0.0

        cosine_value = float(np.dot(a, b) / denominator)

        return float(np.clip(cosine_value, 0.0, 1.0))

#BM25

In [ ]:
pip install rank-bm25

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi


def clean_for_matching(text: str) -> str:
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9+#.\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def tokenize(text: str) -> list[str]:
    cleaned_text = clean_for_matching(text)
    return cleaned_text.split()


class BM25Matcher:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.name = "BM25"

    def score(self, jd_text: str, resume_texts: list[str]) -> list[float]:
        if not resume_texts:
            return []

        tokenized_resumes = [tokenize(text) for text in resume_texts]
        tokenized_jd = tokenize(jd_text)

        if not tokenized_jd or not any(tokenized_resumes):
            return [0.0 for _ in resume_texts]

        bm25 = BM25Okapi(
            tokenized_resumes,
            k1=self.k1,
            b=self.b
        )

        raw_scores = bm25.get_scores(tokenized_jd)

        # Normalize score về khoảng 0-1 để dễ so sánh
        max_score = np.max(raw_scores)

        if max_score <= 0:
            return [0.0 for _ in resume_texts]

        normalized_scores = raw_scores / max_score

        return np.clip(normalized_scores, 0.0, 1.0).astype(float).tolist()

In [ ]:
jd_text = """
Machine Learning Engineer with 3 years experience. Required Python SQL scikit-learn PyTorch natural language processing LangChain REST API Docker AWS data visualization bachelor computer science AWS certification.
"""

resume_texts = [
    """
    Machine learning engineer with 4 years experience. Skills Python SQL PyTorch natural language processing LangChain Docker AWS scikit-learn data visualization. Bachelor Computer Science. AWS Certified.
    """
]

matcher = BM25Matcher()
scores = matcher.score(jd_text, resume_texts)

for i, score in enumerate(scores, start=1):
    print(f"Resume {i}: {score:.4f} = {score * 100:.2f}%")

Resume 1: 0.0000 = 0.00%


## System Architecture

The project has two related paths:

1. Ranking path: parse documents, clean text, score resumes against the job description, compute hybrid score, and rank candidates.
2. RAG preparation path: chunk documents, embed chunks, store vectors in Qdrant, and retrieve relevant chunks for grounded explanation prompts.

Qdrant is not the final ranking model. It is used as a vector database for retrieval evidence.

```text
Resume/JD files
  -> Document parsing
  -> Text cleaning and chunking
  -> Matching models: TF-IDF, BM25, SBERT, E5, BGE, Google Embeddings
  -> Hybrid score: semantic + skills + experience + education
  -> Ranked candidates and explanations

Cleaned chunks
  -> Embedding model
  -> Qdrant vector database
  -> Top-k retrieval
  -> RAG context for LLM explanation or HR question answering
```


## Load Sample Job Description and Resumes

The default demo uses one machine learning engineer job description and three sample resumes.


In [ ]:
from config.settings import SAMPLE_JD_DIR, SAMPLE_RESUME_DIR
from src.preprocessing.document_parser import DocumentParser
from src.services.matching_service import ResumeDocument

parser = DocumentParser()
jd_files = sorted(SAMPLE_JD_DIR.glob('*.txt'))
resume_files = sorted(SAMPLE_RESUME_DIR.glob('*.txt'))

if not jd_files or not resume_files:
    raise FileNotFoundError('Sample files are missing. Check data/sample_job_descriptions and data/sample_resumes.')

jd_text = parser.parse_path(jd_files[0]).cleaned_text
resume_docs = [
    ResumeDocument(candidate_id=path.stem, filename=path.name, text=parser.parse_path(path).cleaned_text)
    for path in resume_files
]

print('Job description:', jd_files[0].name)
print('Resumes:', [doc.filename for doc in resume_docs])
print('\nJD preview:\n', jd_text[:700])


ModuleNotFoundError: No module named 'config'

## Run Resume Matching

Use `tfidf` or `bm25` first for a quick run. Semantic models such as `sbert`, `e5`, and `bge` may download model weights on first use.


In [ ]:
import pandas as pd
from src.services.matching_service import match_resumes

model_key = 'tfidf'  # Try: 'tfidf', 'bm25', 'sbert', 'e5', 'bge'
results = match_resumes(jd_text=jd_text, resumes=resume_docs, model_key=model_key, use_hybrid_score=True)

ranking_table = pd.DataFrame([result.to_export_dict() for result in results])
display_columns = [
    'rank',
    'filename',
    'model_name',
    'final_score',
    'semantic_score',
    'recommendation',
    'matched_skills',
    'missing_skills',
]
ranking_table[display_columns]


ModuleNotFoundError: No module named 'src'

## Compare Fast Baseline Models

This cell compares TF-IDF and BM25 on the sample data. These models are useful baselines for the thesis because they are transparent and fast.


In [ ]:
comparison_rows = []
for key in ['tfidf', 'bm25']:
    model_results = match_resumes(jd_text=jd_text, resumes=resume_docs, model_key=key, use_hybrid_score=True)
    for result in model_results:
        comparison_rows.append(
            {
                'model': key,
                'rank': result.rank,
                'candidate': result.filename,
                'final_score': result.final_score,
                'semantic_or_model_score': result.semantic_score,
                'recommendation': result.recommendation,
            }
        )

pd.DataFrame(comparison_rows).sort_values(['model', 'rank'])


## Generate an Evidence-Based Explanation

This deterministic explanation does not require an API key. If a Gemini API key is configured in `.env`, the project can also call the LLM explanation chain.


In [ ]:
from src.llm.explanation_chain import generate_candidate_explanation

top_candidate = results[0]
explanation = generate_candidate_explanation(
    result=top_candidate,
    jd_text=jd_text,
    resume_text=top_candidate.cleaned_resume_text,
    use_llm=False,
)
print(explanation)


## Qdrant Vector Database for RAG Preparation

This section embeds the job description and resumes, stores chunk vectors in Qdrant, and retrieves relevant chunks for a query.

In Colab, this notebook uses Qdrant local embedded mode through `qdrant-client`, so no Docker server is required.

The embedded dataset in this demo is:

- `data/sample_job_descriptions/machine_learning_engineer.txt`
- `data/sample_resumes/candidate_alex_python_ml.txt`
- `data/sample_resumes/candidate_brianna_frontend.txt`
- `data/sample_resumes/candidate_chris_data_analyst.txt`


In [ ]:
from src.services.vector_store_service import (
    LocalEmbeddingEncoder,
    QdrantRAGStore,
    build_rag_context,
    build_rag_documents,
)

# all-MiniLM is small and fast for Colab demos. For thesis experiments, E5 or BGE can also be used.
rag_encoder = LocalEmbeddingEncoder(model_name='sentence-transformers/all-MiniLM-L6-v2')
rag_store = QdrantRAGStore(collection_name='resume_rag_chunks_colab', encoder=rag_encoder)

rag_documents = build_rag_documents(jd_text, resume_docs)
indexed_count = rag_store.index_documents(rag_documents)
print(f'Indexed {indexed_count} chunks into Qdrant collection resume_rag_chunks_colab')


ModuleNotFoundError: No module named 'src'

In [ ]:
query = 'Which candidates have Python, NLP, machine learning, Docker, and cloud experience?'
retrieved_chunks = rag_store.search(query=query, limit=5, document_type='resume')

retrieval_table = pd.DataFrame(
    [
        {
            'score': round(item.score, 4),
            'document_type': item.document_type,
            'filename': item.filename,
            'chunk_id': item.chunk_id,
            'preview': item.text[:250],
        }
        for item in retrieved_chunks
    ]
)
retrieval_table


In [ ]:
rag_context = build_rag_context(retrieved_chunks, max_chars=3500)
print(rag_context)


## RAG Prompt Template

The retrieved Qdrant chunks can be inserted into a prompt. This keeps the LLM grounded in evidence from the uploaded resumes and job description.


In [ ]:
rag_prompt = f'''
You are an HR screening assistant.
Use only the retrieved evidence below.
Do not infer protected attributes or make final hiring decisions.

Question:
{query}

Retrieved evidence from Qdrant:
{rag_context}

Answer format:
1. Short answer
2. Evidence by candidate
3. Missing or weak evidence
4. Suggested HR follow-up questions
'''.strip()

print(rag_prompt)


## Optional Evaluation Pipeline

If `data/processed/test_pairs.csv` exists, this cell runs the benchmark pipeline. If it does not exist, prepare the dataset pairs first using the project script.


In [ ]:
from config.settings import PROCESSED_DATA_DIR
from src.evaluation.benchmark import run_benchmark

pairs_path = PROCESSED_DATA_DIR / 'test_pairs.csv'
if pairs_path.exists():
    benchmark = run_benchmark(pairs_path=pairs_path, model_keys=['tfidf', 'bm25'], limit_jobs=20)
    display(benchmark)
else:
    print('No processed test pairs found at:', pairs_path)
    print('Prepare pairs with:')
    print('python -m src.data.prepare_pairs --primary-pairs data/raw/<ranking_dataset_file>.csv')


## Optional Streamlit Demo in Colab

Colab is mainly for experiments and notebooks, but you can expose the Streamlit UI with a tunnel. This is optional and may require extra setup.

Example using localtunnel:

```python
# !npm install -g localtunnel
# !streamlit run app.py --server.port 8501 & npx localtunnel --port 8501
```


## Thesis Summary

This project implements an explainable resume-job matching assistant. The system parses resumes and job descriptions, cleans the text, compares candidates using baseline IR models and embedding models, and computes an interpretable hybrid score. The hybrid score combines semantic similarity, required skill overlap, experience evidence, and education/certification evidence.

Qdrant improves the thesis by adding a vector database layer. Instead of only calculating similarity at runtime, the system can persist embeddings for job descriptions and resumes as searchable chunks. This makes the system RAG-ready: when HR asks a question, the system retrieves the most relevant resume/JD chunks from Qdrant and uses them as grounded context for an LLM explanation.

The final design separates scoring from generation. Ranking scores are produced by deterministic NLP models, while the LLM is constrained to explain or summarize evidence that was already retrieved or computed. This supports transparency, reproducibility, and safer use in HR screening.
